# Triple-point position tracking and classification

This replaces the K-means criterion with **vertical TP position**. Original sensor y increases downward:

- **Candidate 1:** physically lower TP, `y > training_mean + training_SD`.
- **Candidate 2:** average band, `training_mean - training_SD <= y <= training_mean + training_SD`.
- **Candidate 3:** physically higher TP, `y < training_mean - training_SD`.

The mean and population SD (`ddof=0`) use training frames only and remain fixed for validation/test. These are position bins, not evidence of three discrete flow regimes.

Run this notebook from the updated Hemisphere-QML project using your Anaconda kernel. The first 20 original TIFFs are clicked in a desktop window; coordinates go into CSV and the TIFFs stay unchanged. Automatic template tracking must be checked on your actual images. Similarity scores are not calibrated probabilities.

## 1. Imports and parameters

In [ ]:
%matplotlib inline
from pathlib import Path
import sys, json, shutil
import matplotlib.pyplot as plt
from IPython.display import display, Image

PROJECT_DIR = Path.cwd().resolve()
if not (PROJECT_DIR / "schlieren_qml" / "triple_point.py").is_file():
    raise FileNotFoundError("Set PROJECT_DIR to the updated Hemisphere-QML project folder.")
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
from schlieren_qml.triple_point import launch_tp_annotation, track_tp, tp_categories, tp_gallery
from schlieren_qml.notebook import resolve_notebook_config, run_from_notebook, display_saved_figures

IMAGE_DIRECTORY = Path(r"D:\Schlieren\Mach2_run01")  # CHANGE THIS.
START_FRAME = 0
FRAME_LIMIT = 20000
FPS = 51032.0
POD_ROI = None  # Original-pixel [x0,y0,x1,y1); include the whole TP excursion.
POD_BIN_FACTOR = 2
MODE_COLOR_VMAX = 0.008

TP_WORK_DIRECTORY = PROJECT_DIR / "results" / "tp_measurement"
ANNOTATION_CSV = TP_WORK_DIRECTORY / "tp_manual.csv"
TRACK_DIRECTORY = TP_WORK_DIRECTORY / "tracking"
CATEGORY_DIRECTORY = TP_WORK_DIRECTORY / "categories"
TRACKS_CSV = TRACK_DIRECTORY / "tp_tracks.csv"
MANUAL_COUNT = 20
DESKTOP_BACKEND = "TkAgg"  # Windows/Anaconda; QtAgg is an alternative if installed.

# Tracking parameters are starting values, requiring validation on real images.
PATCH_SIZE = 25  # Odd side length in ORIGINAL sensor pixels.
SEARCH_RADIUS = 20  # Local search around last accepted TP, in original pixels.
TP_SEARCH_ROI = None  # Optional fixed original-pixel bounds covering all TP motion.
MIN_TRACK_SCORE = 0.75
MIN_PEAK_MARGIN = 0.05
MAX_TEMPLATES = 5
OVERWRITE_TRACKS = False
OVERWRITE_CATEGORIES = False

# Enable one stage at a time after checking the preceding stage.
OPEN_ANNOTATION_WINDOW = False
RUN_TRACKING = False
OPEN_CORRECTION_WINDOW = False
EXPORT_TP_CATEGORIES = False
RUN_POD_AND_CLASSIFIERS = False
OVERWRITE_POD_RESULTS = False

config = json.loads((PROJECT_DIR / "configs" / "full.json").read_text())
config.update(image_directory=str(IMAGE_DIRECTORY), start=START_FRAME,
              limit=FRAME_LIMIT, stride=1, fps=FPS, roi=POD_ROI,
              bin_factor=POD_BIN_FACTOR, labels_csv=None)
config["figures"].update(mode_color_vmax=MODE_COLOR_VMAX, mode_color_percentile=None)
print("Annotations:", ANNOTATION_CSV)
print("Tracking outputs:", TRACK_DIRECTORY)
print("Seed templates come only from the initial manual training images.")

## 2. Click the first 20 TPs

Set `OPEN_ANNOTATION_WINDOW = True` in parameters, rerun that cell, then run this cell. Left-click TP, then press **Enter** to save and advance. Use **Backspace** to revisit the preceding frame. Closing the window preserves confirmed positions; running again resumes missing annotations. Do not draw directly on TIFF copies or mix marked copies into the input sequence.

Coordinates are in the original full-resolution image even when POD later uses a crop or binning. The desktop window uses the same Python environment in a separate process, leaving inline notebook plotting intact. On this remote Linux environment the actual Windows desktop cannot be checked.

In [ ]:
if OPEN_ANNOTATION_WINDOW:
    launch_tp_annotation(config, ANNOTATION_CSV, count=MANUAL_COUNT, backend=DESKTOP_BACKEND)
else:
    print("Set OPEN_ANNOTATION_WINDOW = True to click the initial TP positions.")

## 3. Track every selected image

Set `RUN_TRACKING = True` after all initial annotations are saved. The tracker matches fixed normalized-intensity patches around the manually marked TP. It is a starting measurement method, not a validated geometric TP detector. The first 20 consecutive frames may show only part of an oscillation; inspect examples throughout the record and near both extremes.

`accepted=False` means low similarity or an ambiguous peak. Such a match does not update the tracking anchor. The CSV still records the proposed location so it can be inspected. No final category labels are exported while uncertain rows remain. Preserve previous runs with a fresh work folder, or explicitly enable `OVERWRITE_TRACKS` after correcting annotations.

In [ ]:
if RUN_TRACKING:
    tracking = track_tp(
        config, ANNOTATION_CSV, TRACK_DIRECTORY,
        manual_count=MANUAL_COUNT, patch_size=PATCH_SIZE,
        search_radius=SEARCH_RADIUS, search_roi=TP_SEARCH_ROI,
        min_score=MIN_TRACK_SCORE, min_margin=MIN_PEAK_MARGIN,
        max_templates=MAX_TEMPLATES, overwrite=OVERWRITE_TRACKS,
    )
    display({k:tracking[k] for k in ("n_frames", "n_uncertain", "tracks_csv", "limitation")})
else:
    print("Set RUN_TRACKING = True after finishing the manual seed annotations.")

## 4. Inspect overlays and correct flagged positions

The orange cross marks TP on an original image. Change the seed to inspect other frames. Verify even accepted matches; a high correlation can still select the wrong shock feature. For flagged examples, set `FLAGGED_ONLY=True`.

Set `OPEN_CORRECTION_WINDOW=True` and run the correction cell to click uncertain frames. The orange proposed marker is only a suggestion: click the correct TP and press Enter. These new manual points are position anchors; they do not become new appearance templates. Then rerun tracking with `OVERWRITE_TRACKS=True`. Confirmed correction points are retained in the annotation CSV. Annotation on held-out frames is measurement/label review, not permission to tune the classifiers using test accuracy.

In [ ]:
FLAGGED_ONLY = False
OVERLAY_SEED = 42
if TRACKS_CSV.is_file():
    stored_config = json.loads((TRACK_DIRECTORY / "tp_tracking_manifest.json").read_text())["config"]
    overlays = tp_gallery(stored_config, TRACKS_CSV, examples=9,
                          seed=OVERLAY_SEED, flagged_only=FLAGGED_ONLY)
    display(overlays["figure"])
    plt.close(overlays["figure"])
else:
    print("No saved TP tracking yet.")

In [ ]:
if OPEN_CORRECTION_WINDOW:
    stored_config = json.loads((TRACK_DIRECTORY / "tp_tracking_manifest.json").read_text())["config"]
    launch_tp_annotation(stored_config, ANNOTATION_CSV, count=MANUAL_COUNT,
                         review_csv=TRACKS_CSV, backend=DESKTOP_BACKEND)
else:
    print("Set OPEN_CORRECTION_WINDOW = True to correct uncertain tracked points.")

## 5. Freeze position boundaries and export the three labels

Set `EXPORT_TP_CATEGORIES=True` only after overlay inspection and corrections. This stage uses the chronological training block and gaps recorded in the tracking configuration. It does not estimate the mean/SD from validation or test frames. The two boundaries belong to category 2. The vertical convention is explicit: candidate 1 is lower in the image (larger y), candidate 3 is higher (smaller y).

The export includes `tp_labels.csv`, all TP positions with labels, the numerical category definition, a position-time figure, and a classifier configuration using these external labels. No K-means assignment is used.

In [ ]:
if EXPORT_TP_CATEGORIES:
    categories = tp_categories(TRACKS_CSV, CATEGORY_DIRECTORY, overwrite=OVERWRITE_CATEGORIES)
    display(categories)
    display(Image(filename=str(CATEGORY_DIRECTORY / "tp_position_categories.png")))
else:
    print("Set EXPORT_TP_CATEGORIES = True after validating the TP measurements.")

## 6. Inspect original images in each TP category

In [ ]:
states_path = CATEGORY_DIRECTORY / "tp_states.csv"
if states_path.is_file():
    stored_config = json.loads((TRACK_DIRECTORY / "tp_tracking_manifest.json").read_text())["config"]
    for label, meaning in [("candidate_1", "Physically lower TP"),
                           ("candidate_2", "Average-position band"),
                           ("candidate_3", "Physically higher TP")]:
        print(label, "—", meaning)
        try:
            group = tp_gallery(stored_config, states_path, examples=3,
                               seed=42, only_category=label)
        except ValueError as exc:
            print(exc)
            continue
        display(group["figure"])
        plt.close(group["figure"])
else:
    print("Export TP categories first.")

## 7. Run POD and SVMs against TP-derived labels

This is a **new experiment** with a different label definition. Its accuracy is not directly comparable to earlier K-means label-reproduction scores. A new output directory preserves previous results. Changing the crop requires checking that the selected POD field still contains information about TP motion.

The run configuration is taken from the saved TP measurement experiment, preserving its frame selection, training/validation/test boundaries, and mean/SD definition. Both classifiers receive the same labels and capped samples. Existing POD/quantum controls are in `configs/full.json` before starting TP measurement. Inspect all acquisition timing assumptions, tracking errors, state support, and repeat-run robustness before publication.

In [ ]:
if RUN_POD_AND_CLASSIFIERS:
    tp_config_path = CATEGORY_DIRECTORY / "tp_run_config.json"
    tp_run_config = resolve_notebook_config(json.loads(tp_config_path.read_text()), PROJECT_DIR)
    tp_run_manifest = run_from_notebook(tp_run_config, PROJECT_DIR, overwrite=OVERWRITE_POD_RESULTS)
    destination = Path(tp_run_config["output_directory"])
    shutil.copyfile(CATEGORY_DIRECTORY / "tp_category_definition.json", destination / "tp_category_definition.json")
    display({key:tp_run_manifest.get(key) for key in ("status", "frames", "classification_status")})
    display_saved_figures(tp_run_manifest, names=["04_spatial_modes", "09_confusion_matrices"], width=1100)
else:
    print("Set RUN_POD_AND_CLASSIFIERS = True after checking TP categories.")

## What remains to validate experimentally

The synthetic software checks verify tracking coordinates, quality gating, sensor-axis direction, and training-only thresholds. They do not establish TP-tracking accuracy on your schlieren sequence. Check manual landmarks distributed through the record, especially extrema, low-contrast frames, and apparent tracking jumps. Template size, search range, and score thresholds need an independently checked measurement procedure. Document labeling uncertainty near each category boundary. Model choices belong to validation; use a fresh final holdout for the new experiment.